# 02 · Train Model A on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bileldh-ing/PrintGuard-/blob/main/notebooks/02_train_model_a_colab.ipynb)

**Runtime → Change runtime type → GPU (T4 or better).**

Steps: clone the repo → install → download + prepare data (`PROFILE=full`) → train the YOLO stringing
detector and the YOLO frame classifier → evaluate on the held-out test split → export to ONNX →
download the weights.

Full data is tens of GB. Mount Google Drive to keep `data/` and `weights/` between sessions, or start
with `PROFILE = "smoke"` to check that everything runs.

In [ ]:
!nvidia-smi || echo 'No GPU - switch the runtime to GPU'

In [ ]:
USE_DRIVE = False   # True: keep data/ and weights/ on Google Drive across sessions
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    %cd /content/drive/MyDrive

In [ ]:
# Clone the repo (skipped when the notebook already runs inside it)
import os, subprocess, sys
if not os.path.exists("printguard/__init__.py"):
    if not os.path.exists("PrintGuard-"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bileldh-ing/PrintGuard-.git"], check=True)
    os.chdir("PrintGuard-")
print(os.getcwd())

In [ ]:
PROFILE = "full"    # "smoke" for a quick end-to-end check
DEVICE = "0"        # first CUDA GPU
!pip -q install -r requirements-train.txt
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## Data (resumable, checksummed)

In [ ]:
!make data PROFILE={PROFILE}

## Train the detector and the classifier

In [ ]:
SMOKE = PROFILE == "smoke"
det_override = "--override configs/smoke/model_a_detect.yaml" if SMOKE else ""
cls_override = "--override configs/smoke/model_a_classify.yaml" if SMOKE else ""
!python -m printguard.training.train_a --config configs/model_a_detect.yaml {det_override} --device {DEVICE}
!python -m printguard.training.train_a --config configs/model_a_classify.yaml {cls_override} --device {DEVICE}

## Export to ONNX (torch-free inference)

In [ ]:
!make export PROFILE={PROFILE}

## Evaluate on the test split

In [ ]:
WDIR = "weights/smoke" if SMOKE else "weights"
RDIR = "results/smoke" if SMOKE else "results"
IMG = 256 if SMOKE else 640
!python -m printguard.evaluation.eval_a classifier --weights {WDIR}/model_a_classify.onnx --out {RDIR}/model_a
!python -m printguard.evaluation.eval_a detector --weights {WDIR}/model_a_detect.pt --imgsz {IMG} --out {RDIR}/model_a

In [ ]:
import json
for f in ["classifier_test.json", "detector_test.json"]:
    d = json.load(open(f"{RDIR}/model_a/{f}"))
    print(f, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in d.items()
              if k not in ("per_class", "confusion_matrix", "speed_ms_per_image")})
from IPython.display import Image
Image(f"{RDIR}/model_a/classifier_test_confusion.png")

## Download weights and results

In [ ]:
!zip -qr model_a_artifacts.zip {WDIR}/*.onnx {WDIR}/*.pt {RDIR}/model_a
try:
    from google.colab import files
    files.download("model_a_artifacts.zip")
except ImportError:
    print("saved model_a_artifacts.zip")

**Next:** `03_train_model_b_colab.ipynb` (needs the Model A weights from this notebook).
When you report numbers, include the profile, GPU, epochs and commit hash.